In [ ]:
import stereo as st
import warnings
warnings.filterwarnings("ignore")

import sys
import os
import shutil   # ✅ needed for copytree
from pathlib import Path
from natsort import natsorted

In [ ]:
home_dir = Path.home()

### GCP
src_data_dir = home_dir / 'ext_hd_sammy' / 'data' / 'stomics' / 'gene_exp'
dst_folder   = home_dir / 'ext_hd_sammy' / 'projects' / 'out' / 'out_stomics' / 'script01_output'

### HPC (uncomment if running there)
# src_data_dir = home_dir / 'scratch' / 'data' / 'stomics' / 'gene_exp'
# dst_folder   = home_dir / 'scratch' / 'projects' / 'sammy' / 'out' / 'out_stomics' / 'script01_output'

os.makedirs(dst_folder, exist_ok=True)

# ---------------------- SUBSET ISOLATION ----------------------
subset_dir = home_dir / 'ext_hd_sammy' / 'projects' / 'worksets' / 'stomics_subset_so40_64_59_44_45_46'
subset_dir.mkdir(parents=True, exist_ok=True)

# Only process these 6 Chip IDs, no copying
chip_ids = [
    "C04143D3", "B04101E6", "C04143G2",
    "B04101A3", "A04100A6", "C04139D3"
]

stomics_data_folders = chip_ids
print(f"Will scan ONLY these {len(stomics_data_folders)} chips: {stomics_data_folders}")



In [ ]:
stomics_data_folders

In [ ]:
dst_dir_cellbin = dst_folder / 'cellbin'
os.makedirs(dst_dir_cellbin, exist_ok=True)

dst_dir_cellbin_adjusted = dst_folder / 'cellbin_adjusted'
os.makedirs(dst_dir_cellbin_adjusted, exist_ok=True)

dst_dir_ssDNA = dst_folder / 'ssDNA'
os.makedirs(dst_dir_ssDNA, exist_ok=True)


# ---------------------- MAIN LOOP ----------------------
for folder in stomics_data_folders:
    folder_path = str(src_data_dir / folder / '03.ssDNA_analysis')
    if not os.path.exists(folder_path):
        print(f"Folder {folder_path} does not exist. Skipping.")
        continue

    ### using cellbin.gef files
    if not [f for f in os.listdir(folder_path) if f.endswith('.cellbin.gef')]:
        print(f"No .cellbin.gef files found in {folder_path}. Skipping.")
        continue
    
    
    print(f"Processing folder: {folder_path}")
    filename = [f for f in os.listdir(folder_path) if f.endswith('.cellbin.gef')][0]
    print(f"Processing file: {folder_path}/{filename}")
    data = st.io.read_gef(f'{folder_path}/{filename}', bin_type='cell_bins')
    
    adata_filename = f"{folder}_cellbin.h5ad"
    print(f"Saving to {dst_dir_cellbin}/{adata_filename}")
    adata = st.io.stereo_to_anndata(data, flavor='scanpy', output= f'{dst_dir_cellbin / adata_filename}')

    ### using adjusted.cellbin.gef files
    if not [f for f in os.listdir(folder_path) if f.endswith('.adjusted.cellbin.gef')]:
        print(f"No .adjusted.cellbin.gef files found in {folder_path}. Skipping.")
        continue
    filename_adjusted = [f for f in os.listdir(folder_path) if f.endswith('.adjusted.cellbin.gef')][0]
    print(f"Processing file: {folder_path}/{filename_adjusted}")
    data_adjusted = st.io.read_gef(f'{folder_path}/{filename_adjusted}', bin_type='cell_bins')
    adata_adjusted_filename = f"{folder}_cellbin_adjusted.h5ad"
    print(f"Saving to {dst_dir_cellbin_adjusted}/{adata_adjusted_filename}")
    adata_adjusted = st.io.stereo_to_anndata(data_adjusted, flavor='scanpy', output= f'{dst_dir_cellbin_adjusted / adata_adjusted_filename}')
    
    
    '''
    ### using ssDNA.gef files
    if not [f for f in os.listdir(folder_path) if f.endswith('.ssDNA.gef')]:
        print(f"No .ssDNA.gef files found in {folder_path}. Skipping.")
        continue
    filename_ssDNA = [f for f in os.listdir(folder_path) if f.endswith('.ssDNA.gef')][0]
    print(f"Processing file: {folder_path}/{filename_ssDNA}")
    data_ssDNA = st.io.read_gef(f'{folder_path}/{filename_ssDNA}', bin_type='bins', bin_size=100)
    adata_ssDNA_filename = f"{folder}_ssDNA.h5ad"
    print(f"Saving to {dst_dir_ssDNA}/{adata_ssDNA_filename}")
    adata_ssDNA = st.io.stereo_to_anndata(data_ssDNA, flavor='scanpy', output= f'{dst_dir_ssDNA / adata_ssDNA_filename}')
    '''

print("All selected Stomics data processed and saved to the output folder.")